# PREDICT/90 Pro - Football Match Predictor

Home / Draw / Away probabilities for **38 leagues** and **national teams**, plus Over/Under 2.5, BTTS, value bets, Kelly stakes,
corners, cards, shots, 4 daily accas, predictions by strength, injuries and a bet tracker. Every number below is built only from information known **before kickoff**.

**Run it:** menu *Run -> Run All* (Jupyter) or *Runtime -> Run all* (Google Colab).

| Part | What you get |
|---|---|
| 1 | Setup (PC or Colab) |
| 2 | Today's and tomorrow's predictions -> also saved to Excel |
| 3 | Injuries and lineups |
| 4 | Backtest: how good is it really, season by season |
| 5 | Goals, corners, cards and shots |
| 5b | National teams: the test |
| 5c | Bangers by market: the test |
| 6 | Value bets and profit vs bookmakers |
| 7 | The accas - tested on the past |
| 8 | Your bet tracker |
| 9 | How it works and why each part was chosen |
| 10 | Tune the models again (optional) |
| 11 | Honest notes, and what differs from the plan |

Data: football-data.co.uk (results + odds, free), API-Football (fixtures, odds, injuries, lineups - free plan).
Models: Logistic Regression + XGBoost + LightGBM + Neural Network, stacked, tuned with Optuna.

## 1. Setup

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Colab: the newest code comes from GitHub, your own files (tracker, models, key, settings) stay in your Google Drive
    import glob, shutil, zipfile, urllib.request, importlib.util, importlib.metadata
    CODE_URL = "https://github.com/RicardoJose000/predict90-colab/archive/refs/heads/main.zip"
    if int(importlib.metadata.version("pandas").split(".")[0]) >= 3:          # the code is made for pandas 2
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas>=2.2.3,<3"], check=True)
        print("Colab restarts once now. Wait a few seconds, then press Runtime -> Run all again.")
        os.kill(os.getpid(), 9)
    missing = [p for p in ("optuna", "openpyxl", "lightgbm", "xgboost") if importlib.util.find_spec(p) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    PROJECT = "/content/Football_Predictor_Pro"
    if not os.path.isdir(os.path.join(PROJECT, "p90")):
        print("Downloading the code ...")
        urllib.request.urlretrieve(CODE_URL, "/content/code.zip")
        shutil.rmtree("/content/code", ignore_errors=True)
        with zipfile.ZipFile("/content/code.zip") as z:
            z.extractall("/content/code")
        shutil.move(glob.glob("/content/code/*")[0], PROJECT)
    FILES = "/content/my_files"
    try:
        from google.colab import drive
        drive.mount("/content/drive")      # Colab asks you to allow Google Drive: press Connect / Allow
        FILES = "/content/drive/MyDrive/Football_Predictor_files"
    except Exception as e:
        print(f"Google Drive is not connected ({e}). It still works, but the tracker and models are lost when Colab closes.")
    for f in ("settings.json", "models/best_params.json", "output/backtest_predictions.csv.gz",
              "output/backtest_more_bets.csv.gz", "output/combination_test.csv", "output/backtest_internationals.csv.gz"):
        src, dst = os.path.join(PROJECT, f), os.path.join(FILES, f)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.exists(src) and (not os.path.exists(dst) or (f.startswith("output/") and os.path.getsize(src) != os.path.getsize(dst))):
            shutil.copy(src, dst)          # your settings are kept, the test results are always the newest
    os.environ["P90_FILES"] = FILES
    os.chdir(PROJECT)
sys.path.insert(0, os.getcwd())

API_FOOTBALL_KEY = ""   # optional: paste your free API-Football key here ONE time - it is saved to .env
if API_FOOTBALL_KEY:
    open(os.path.join(os.environ.get("P90_FILES", "."), ".env"), "w").write(f"API_FOOTBALL_KEY={API_FOOTBALL_KEY.strip()}\n")

import numpy as np, pandas as pd
from IPython.display import display, Markdown
from p90 import pipeline, evaluate, charts, config, props, tracker, live, intl, bangers
from p90.config import MAIN_LEAGUES, league_label

S = config.load_settings()
print(f"Folder : {os.path.basename(os.getcwd())} | your files: {config.FILES.name}")
print(f"Leagues: {len(S['leagues'])}")
print(f"API-Football key: {'found' if config.api_key() else 'not set (injuries + extra fixtures are skipped)'}")
import numpy, scipy, sklearn, lightgbm, xgboost
print(f"Versions: Python {sys.version.split()[0]}, pandas {pd.__version__}, numpy {numpy.__version__}, scikit-learn {sklearn.__version__}, LightGBM {lightgbm.__version__}, XGBoost {xgboost.__version__}")

NAMES = {"fold": "Season", "matches": "Matches", "accuracy": "Model", "accuracy_bookie": "Bookmaker", "logloss": "Log-loss",
         "logloss_bookie": "Log-loss bookmaker", "conf_acc": "Picks 50%+", "conf_share": "Share 50%+", "strong_acc": "Picks 65%+",
         "strong_share": "Share 65%+", "strong_n": "Number 65%+", "threshold": "At least this sure", "share": "Share of matches",
         "predicted": "Model said", "actual": "Really happened", "n": "Outcomes", "bin": "Range", "league": "Code", "name": "League",
         "min_edge": "Min edge", "bets": "Bets", "hit": "Hit rate", "avg_odds": "Avg odds", "roi": "ROI", "slips": "Slips",
         "break_even": "Break-even", "said": "Model said", "real": "Really happened", "legs": "Legs", "pick": "Bet"}
PCT = {"Home %", "Draw %", "Away %", "Pick %", "Over 2.5 %", "BTTS %", "Edge %", "Model", "Bookmaker", "Picks 50%+", "Share 50%+",
       "Picks 65%+", "Share 65%+", "At least this sure", "Share of matches", "Model said", "Really happened", "Min edge",
       "Hit rate", "ROI", "Break-even", "Accuracy", "Always pick the most common", "65%+ picks accuracy",
       "Happened", "Share", "Right", "Model said (60%+)", "Always likelier side", "Lowest season",
       "Bet365", "Best odds", "2018-21", "2022 on", "Higher rated wins", "Picks 75%+", "Share 75%+", "Share 60%+",
       "Won", "Profit per slip"}
def show(df, n=None):
    """Pretty table: friendly names, percentages as %, odds with 2 decimals."""
    if df is None or len(df) == 0:
        print("(nothing to show)"); return
    d = (df.head(n) if n else df).rename(columns=NAMES)
    fmt = {c: "{:.1%}" for c in d.columns if c in PCT}
    fmt.update({c: "{:.2f}" for c in d.columns if ("odds" in str(c).lower() or c == "Stake") and c not in PCT})
    fmt.update({c: "{:.2f}" for c in d.columns if c in ("Staked", "Profit")})
    fmt.update({c: "{:.4f}" for c in d.columns if "Log-loss" in str(c)})
    fmt.update({c: "{:,.0f}" for c in d.columns if c in ("Matches", "Bets", "Slips", "Outcomes", "Number 65%+", "65%+ picks", "60%+ picks")})
    fmt.update({"Slips a day": "{:.1f}"} if "Slips a day" in d.columns else {})
    fmt.update({c: "{:,.0f}" for c in ("Picks", "Per season", "Real odds slips") if c in d.columns})
    fmt.update({"Legs (avg)": "{:.1f}"} if "Legs (avg)" in d.columns else {})
    fmt.update({"Expected": "{:.1f}"} if "Expected" in d.columns else {})
    fmt.update({"Legs": "{:.1f}"} if "Legs" in d.columns and d["Legs"].dtype == float else {})
    fmt.update({"Pick %": "{:.1%}"} if "Pick %" in d.columns else {})
    fmt = {c: f for c, f in fmt.items() if pd.api.types.is_numeric_dtype(d[c])}   # text columns stay as they are
    display(d.style.format(fmt, na_rep="").hide(axis="index"))

## 2. Today's and tomorrow's predictions

One call does everything: download the newest results and odds -> upcoming matches -> features ->
models (trained again automatically every 7 days) -> probabilities -> value bets -> accas -> Excel file in `output/`.

In [ ]:
out = pipeline.daily()
T = pipeline.tables(out) if len(out.get('predictions', [])) else {}

### The results page

The same results as the Excel file, as one page with tabs and filters. It works on a phone, right here in the notebook.
It is also saved in the `output` folder as `predictions_...html`: on a PC it opens by itself after `RUN_ME.bat`.

In [ ]:
if out.get("page"):
    from html import escape
    from IPython.display import HTML
    display(HTML('<div><iframe srcdoc="' + escape(out["page"].read_text(encoding="utf-8")) + '" style="width:100%;height:760px;border:0;border-radius:12px" title="Results page"></iframe></div>'))

### Predictions by strength

**Banger** = pick is 75%+, **Strong** = 65-75%, **Good** = 50-65%, **Weak** = below 50%. The test of each group is in part 4.
In the Excel file and the page `All`, `Bangers`, `Strong` and `Weak` each have their own tab, and you can filter them by day and league, also by `League type` (Main or Other).

In [ ]:
PICK_COLS = ["Date","Time","League","League type","Home","Away","Pick","Pick %","Home %","Draw %","Away %","Odds 1","Odds X","Odds 2"]
print("Bangers (75%+)")
show(T.get("Bangers", pd.DataFrame()).reindex(columns=PICK_COLS))
print("Strong (65-75%)")
show(T.get("Strong", pd.DataFrame()).reindex(columns=PICK_COLS))

### Value bets
Our chance x the odds beats 1.05. *Stake* = quarter-Kelly on the bankroll in `settings.json`.

In [ ]:
show(T.get("Value bets"))

### My picks

My own recommendation: 1 long acca around 5 odds and up to 2 safer accas around 2.3 odds, only when the model has them. No match is used twice. Part 7 has the test of this exact rule: about break-even, not a sure profit.

In [ ]:
show(T.get("My picks"))

### Bangers by market

The safest pick of each kind, only 75%+: 1X2, Over 2.5, Over 1.5, corners, cards and shots. A match in Over 2.5 is not shown again in Over 1.5.
Corners, cards and shots show every match once, at its highest line that is still 75%+. No BTTS: the model never reaches 75% for it.
Then one acca per market and day (total odds about 2 to 3): one market per slip, so the same match is never twice in a slip. Part 5c has the test.

In [ ]:
for name in ["1X2", "Over 2.5 goals", "Over 1.5 goals", "Corners over", "Cards over", "Shots over"]:
    t = T.get(name, pd.DataFrame())
    print(f"{name}: {len(t)} picks")
    show(t, 10)
print("Banger accas")
show(T.get("Banger accas"))

### The 4 accas

The `SLIP` row shows total odds and the chance that every leg wins. Within one sheet every match is used once. Part 7 has the test of each.

**Main acca**: win, double chance and over/under 2.5. These have old odds, so their profit is tested.

In [ ]:
show(T.get("Main acca"))

**Strong acca**: favourites to win only, every leg 65%+. Only 1 or 2 a day and only on some days. A small profit in the test, not proven.

In [ ]:
show(T.get("Strong acca"))

**Banger acca**: favourites to win only, every leg 70%+. Very few, about 40 a season. A small profit in the test, not proven: read part 7 first.

In [ ]:
show(T.get("Banger acca"))

**Other acca**: bets with no old odds, so accuracy is tested but profit is not. `Price = estimate` means no real odds were found: only bets 60%+ sure are used then. Check the real price at your bookmaker before you bet.

In [ ]:
show(T.get("Other acca"))

**International acca**: national teams, favourites to win only, competitive matches only (no friendlies), every leg 70%+. Their accuracy is tested in part 5b. Their profit is not: the free data has no old odds for national teams.

In [ ]:
show(T.get("International acca"))

### National teams

World Cup and Euro qualifiers, Nations League, AFCON, Copa America, Gold Cup, Asian Cup and friendlies, today and tomorrow (needs the API-Football key).
They have their own model, because the club models learned from club leagues. **Friendlies are always WEAK**: squads change a lot there. The test is in part 5b.

In [ ]:
show(T.get("Internationals", pd.DataFrame()).reindex(columns=PICK_COLS + ["Strength", "Over 2.5 %", "BTTS %"]))

### All matches

In [ ]:
cols = ["Date","Time","League","League type","Home","Away","Home %","Draw %","Away %","Pick","Strength","Odds 1","Odds X","Odds 2","Over 2.5 %","BTTS %","Value bet"]
show(T.get("All", pd.DataFrame()).reindex(columns=cols))

## 3. Injuries and lineups

Injuries come from API-Football for today and tomorrow (free plan).
Lineups are published about 20-40 minutes before kickoff - set `CHECK_LINEUPS = True` and run the cell again close to kickoff
to see the starting XI of every acca leg (1 request per match).

In [ ]:
show(T.get("Injuries"))
CHECK_LINEUPS = False
if CHECK_LINEUPS and len(out.get("predictions", [])):
    api = live.ApiFootball()
    p = out["predictions"]
    legs = {i for name in ("slips", "strong_slips", "banger_slips", "other_slips") for day in out[name].values() for sl in day for i, *_ in sl["legs"]}
    for i in legs:
        r = p.loc[i]
        if pd.notna(r.get("fixture_id")):
            lu = live.lineups(api, r["fixture_id"])
            print(f"\n{r['home']} v {r['away']}:", "not published yet" if lu.empty else "")
            if not lu.empty:
                display(lu.groupby("team")["player"].apply(", ".join).to_frame())

## 4. Backtest - how good is it really?

**Walk-forward test:** for every season the models are trained only on the seasons *before* it, then they predict that
season's matches. Nothing from the future is used. This is exactly how the system works live.

The backtest takes about 1 hour, so the saved results are shown. To check them yourself set
`RUN_BACKTEST = True`: with `USE_SNAPSHOT = True` it uses the saved data and fixed random seeds, so you get the same
numbers as here on the library versions in `requirements.txt`. Other versions, like Google Colab's, give about the same picks but can move the acca numbers (part 7). With `USE_SNAPSHOT = False` it uses the newest data (good once a month).
Set `LEAGUE_FILTER = ["E0"]` to see only the Premier League, etc.

In [ ]:
RUN_BACKTEST = False
USE_SNAPSHOT = True
LEAGUE_FILTER = None            # None = all leagues, or for example ["E0", "SP1", "I1", "D1", "F1"]

bt = pipeline.load_backtest()
if bt is None or RUN_BACKTEST:
    bt = pipeline.backtest(use_snapshot=USE_SNAPSHOT)
if LEAGUE_FILTER:
    bt = bt[bt["league"].isin(LEAGUE_FILTER)]
sm = evaluate.summary(bt, S["confident_threshold"], S["strong_threshold"])
print(f"{sm['matches']:,} matches tested, {bt['fold'].nunique()} seasons ({bt['fold'].min()} to {bt['fold'].max()}), {bt['league'].nunique()} leagues")

### The 3 commitments

In [ ]:
rows = [
    ["Picks 50%+ sure", f"{sm['conf_share']:.0%} of matches", sm["conf_acc"], "60%+", "YES" if sm["conf_acc"] >= 0.60 else "no"],
    ["Strong picks 65%+", f"{sm['strong_share']:.0%} of matches", sm["strong_acc"], "65%+", "YES" if sm["strong_acc"] >= 0.65 else "no"],
    ["All matches", "100%", sm["accuracy"], f"aim 55-56% (bookie {sm['accuracy_bookie']:.1%})", "-"],
]
show(pd.DataFrame(rows, columns=["Which matches", "How many", "Accuracy", "Target", "Met"]))

### Bangers, strong, good, weak

The higher the probability, the more often it wins, in main and in other leagues. But the odds go down with it.
*ROI* is betting every pick on its own at average odds: every group loses a little, because of the bookmaker's margin.

In [ ]:
show(evaluate.strength_table(bt, S))

### Season by season

In [ ]:
bs = evaluate.by_group(bt, "fold", S["confident_threshold"], S["strong_threshold"])
charts.season_accuracy(bs);
show(bs[["fold","matches","accuracy","accuracy_bookie","logloss","logloss_bookie","conf_acc","conf_share","strong_acc","strong_share"]])

### Confidence: the more sure the model is, the more often it is right

In [ ]:
curve = evaluate.threshold_curve(bt)
charts.confidence(curve, S["confident_threshold"], S["strong_threshold"]);
show(curve)

### Calibration
When the model says 60%, does it happen about 60% of the time? Dots on the line = honest probabilities.

In [ ]:
cal = evaluate.calibration(bt)
charts.calibration(cal);
show(cal)

### By league
Some leagues are easier to predict than others. Use this to choose where to play.

In [ ]:
bl = evaluate.by_group(bt, "league", S["confident_threshold"], S["strong_threshold"])
charts.league_accuracy(bl, league_label);
bl.insert(1, "name", bl["league"].map(league_label))
show(bl.sort_values("conf_acc", ascending=False)[["league","name","matches","accuracy","accuracy_bookie","conf_acc","conf_share","strong_acc","strong_n"]])

### Is the model better than the bookmaker? (significance test)

Paired bootstrap on log-loss (2,000 resamples of match days) and a sign test on accuracy.
`p_value` below 0.05 would mean a statistically significant edge.

In [ ]:
sig = evaluate.significance(bt)
print(f"Log-loss gain vs bookmaker: {sig['logloss_gain']:+.4f} (95% range {sig['ci_low']:+.4f} to {sig['ci_high']:+.4f}), p = {sig['p_value']:.3f}")
print(f"Matches only the model got right: {sig['acc_only_model']:,} | only the bookmaker: {sig['acc_only_bookie']:,} | p = {sig['acc_p_value']:.3f}")
if sig["p_value"] < 0.05:
    print("-> The model's probabilities are measurably better than the bookmaker's, but the gain is very small.")
    print("   It is not enough on its own to beat the bookmaker's margin of about 6-7%. Parts 6 and 7 show the real profit.")
else:
    print("-> No significant difference from the bookmaker's probabilities on all matches.")

## 5. Goals, corners, cards and shots

In [ ]:
g = evaluate.goals_summary(bt)
rows = [["Over/Under 2.5", g["ou_acc"], g["ou_base"], g["ou_strong_acc"], g["ou_strong_n"]],
        ["Both teams to score", g["btts_acc"], g["btts_base"], g["btts_strong_acc"], g["btts_strong_n"]]]
show(pd.DataFrame(rows, columns=["Market", "Accuracy", "Always pick the most common", "65%+ picks accuracy", "65%+ picks"]))
if "ou_acc_bookie" in g:
    print(f"O/U 2.5 where bookmaker odds exist ({g['ou_mk_matches']:,} matches): model {g['ou_acc_on_mk']:.1%} vs bookmaker {g['ou_acc_bookie']:.1%}, "
          f"log-loss {g['ou_ll_on_mk']:.4f} vs {g['ou_ll_bookie']:.4f}")

### More bets: over 1.5, teams to score, corners, cards, shots

Same walk-forward test, 22 main leagues. **This tests accuracy only.** Old odds for these bets are not in the free data, so their profit can't be tested.

How to read it:
- **Model said / Happened**: on average, did the model's chance match what really happened? Close = honest chances.
- **Right**: when the model was 60%+ sure of one side, how often it was right. **Model said (60%+)** is the chance it gave those picks.
- **Always likelier side**: what you get with no model at all, always picking the more common side. If *Right* is much higher on a line where the two sides are about equal, the model is really adding something. On lopsided lines, like over 1.5, a high *Right* mostly comes from the line itself, and the odds are low for the same reason.

Live, these bets use the half model, half market chance when real odds exist, because the bookmaker prices them too.

In [ ]:
bp = pipeline.load_more_bets()
bp = bp[bp["league"].isin(LEAGUE_FILTER or MAIN_LEAGUES)] if bp is not None else None
if bp is not None and len(bp):
    show(props.accuracy_table(bp))
    print("Corner 1X2: chance only in the Excel file, usually no odds in the free API")
    show(props.corner_1x2_table(bp))

## 5b. National teams: the test

National teams have their own model: Elo ratings for every country (a win against a strong team counts more, World Cup matches count
more than friendlies, the home team gets a bonus) plus recent goals, then logistic regression + LightGBM for home/draw/away and Poisson for goals.
Data: every international result since 1872 (free, updated after each window), saved in `data_snapshot/intl_results.csv.gz`.

**Same walk-forward test:** each year is predicted by a model trained only on the years before it. About 30 seconds, so set `RUN_INTL_BACKTEST = True` to run it yourself.
- **Higher rated wins** = always pick the team with the higher rating. The model gets about the same on all matches: its value is in knowing **how sure** to be.
- No old odds exist for national teams in the free data, so **accuracy is tested, profit is not**. Their odds are usually low, because the bookmakers know the favourites too.

In [ ]:
RUN_INTL_BACKTEST = False
ibt = intl.load_backtest()
if ibt is None or RUN_INTL_BACKTEST:
    ibt = intl.backtest(use_snapshot=USE_SNAPSHOT)
print(f"{len(ibt):,} national team matches tested, years {ibt['fold'].min()} to {ibt['fold'].max()}")
show(intl.accuracy_table(ibt))

### Competitive matches, year by year

In [ ]:
show(intl.season_table(ibt))

### Calibration: when the model says 70%, does it happen about 70% of the time? (competitive matches)

In [ ]:
show(intl.calibration_table(ibt))

### Goals: over/under, BTTS

In [ ]:
show(intl.goals_table(ibt))

## 5c. Bangers by market: the test

The Bangers tab, tested with **exactly the live code** on the walk-forward backtest (each season predicted only with the seasons before it).
1X2 and goals: 38 leagues. Corners, cards and shots: the 22 main leagues. National teams: competitive matches only.
- **Right**: how often the pick won. **Model said**: the average chance it gave. Close = honest chances.
- **Lowest season**: the worst single season. Bangers are not sure things: about 1 in 5 still loses.
- The accas: **Won** = all legs won. Profit only for 1X2, at real old average odds. For the others no old odds exist, so the price is an estimate and profit is not tested.

In [ ]:
bt_all, more_all = pipeline.load_backtest(), pipeline.load_more_bets()
if bt_all is not None:
    show(bangers.accuracy_table(bt_all, more_all, S, intl.load_backtest()))
    show(bangers.acca_table(bt_all, more_all, S))

## 6. Value bets and profit

Flat 1-unit stakes at **average** bookmaker odds (what you can really get), every test season.

In [ ]:
scan = evaluate.edge_scan(bt, S)
show(scan)
r, vb = evaluate.value_backtest(bt, S)
if r["bets"]:
    lo, hi = evaluate.roi_range(vb["profit"], bt.loc[vb.index, "date"])
    print(f"Default rule (edge {S['value_min_edge']:.0%}+): {r['bets']:,} bets, ROI {r['roi']:+.1%}, 95% range {lo:+.1%} to {hi:+.1%}. "
          + ("Proven profit." if lo > 0 else "The range includes zero, so this is not a proven profit."))
rows = [{"Bet on": f"every pick {t:.0%}+", **evaluate.pick_roi(bt, t)} for t in (0.5, 0.6, 0.65, 0.7, 0.75)]
show(pd.DataFrame(rows))

## 7. The accas - tested on the past

Every match day in the backtest, the slips are built exactly like the live system builds them, then settled.
Break-even = 1 / average odds. Which leagues go into slips is set by `acca_leagues` in `settings.json`.

**The main acca is mixed, not only favourites to win:** win, double chance and over/under 2.5. A slip has at most 2 of one type, and a match is used only once.
For over/under the chance is half the model and half the market, because the market is a little better than the model there.

Limits of this test:
- **Double chance prices are worked out from the win odds**, because old double chance odds are not in the free data. Real prices can differ a little. Live, real odds are used when API-Football has them.
- **The other acca is not in this test.** Its old odds are not in the free data, so its profit can't be tested. Its accuracy is in part 5 and below. Your bet tracker logs each acca apart, so it will show how they really do over time.

In [ ]:
acc, slips_bt, days_bt = evaluate.acca_backtest(bt, S)
print(f"{acc['slips']:,} slips on {acc['days']:,} match days | won {acc['won']:,} = {acc['hit']:.1%} "
      f"(break-even {acc['break_even']:.1%}) | avg odds {acc['avg_odds']:.2f}, {acc['avg_legs']:.1f} legs | ROI {acc['roi']:+.1%}")
print(f"Days with all {S['acca_slips']} slips: {acc['full_days']:,} -> at least 1 slip won on {acc['days_1plus']:.0%}, "
      f"at least 2 on {acc['days_2plus']:.0%}, all 5 on {acc['days_5']:.1%}")
slips_bt["fold"] = slips_bt["date"].apply(lambda d: f"{d.year if d.month >= 7 else d.year - 1}/{(d.year + 1 if d.month >= 7 else d.year) % 100:02d}")
per = slips_bt.groupby("fold").agg(slips=("won", "size"), hit=("won", "mean"), avg_odds=("odds", "mean"), roi=("profit", "mean")).reset_index()
per["break_even"] = 1 / per["avg_odds"]
charts.acca_by_season(per);
show(per)
print("Which bets went into the slips, and did they happen as often as the model said?")
show(acc["types"])

### BTTS accuracy
No profit test here, because old BTTS odds don't exist. Only: when the model says it, does it happen? (22 main leagues)

In [ ]:
m = bt[bt["league"].isin(MAIN_LEAGUES)]
rows = []
for th in (0.55, 0.60, 0.65):
    y, n = m[m["p_btts"] >= th], m[m["p_btts"] <= 1 - th]
    rows.append({"Model chance": f"{th:.0%}+", "BTTS yes: matches": len(y), "Model said": y["p_btts"].mean(), "Happened": y["btts"].mean(),
                 "BTTS no: matches": len(n), "Model said (no)": 1 - n["p_btts"].mean(), "Happened (no)": 1 - n["btts"].mean()})
display(pd.DataFrame(rows).style.format({"Model said": "{:.1%}", "Happened": "{:.1%}", "Model said (no)": "{:.1%}", "Happened (no)": "{:.1%}", "BTTS yes: matches": "{:,}", "BTTS no: matches": "{:,}"}).hide(axis="index"))

### Which combination makes a profit?

Every combination of the bet types with old odds, at five leg strengths, best first. **Three made a small profit at average odds, all favourites to win only:
every leg 70%+ (the banger acca), 65%+ (the strong acca) and 60%+. For all three the range still includes a loss, so none is proven.** All the others lost.
More bets gave more slips, not more profit. Stronger favourites did better.

Set `RUN_COMBINATIONS = True` to run this again (a few minutes).

In [ ]:
RUN_COMBINATIONS = False
COMBO_FILE = pipeline.OUTPUT / "combination_test.csv"
if RUN_COMBINATIONS or not COMBO_FILE.exists():
    evaluate.combination_test(pipeline.load_backtest(), S).to_csv(COMBO_FILE, index=False)
show(pd.read_csv(COMBO_FILE))

### Strong and banger accas: how much can you trust them?

Favourites only, at different leg strengths and total odds. Legs of 75%+ almost never reach 5 odds, which is why the banger acca uses 70%+.

**No acca has proven profit. Treat the strong and banger accas as a small profit or about break-even, nothing more.** Reasons, all measured:
- **Acca numbers are very sensitive.** Running this same test, with the same model and data, on other library versions gave the banger acca **+27.6%** instead of **+7.2%**, and the strong acca +5.9% instead of +4.9%. A tiny change swaps a few legs, and a few slips flip between winning and losing.
- **The range includes a loss** for both, and they are not statistically significant (below).
- **They are the best of many combinations**, which makes them look better than they are.
- A few wins make a big part of the profit, there are only about 40 banger slips a season, and some seasons lose.

The picks are much steadier than the accas: the same change moved picks 50%+ by 0.1 points and picks 65%+ by less than 0.1. Your bet tracker logs each acca apart, so use small stakes and let your own record decide.

In [ ]:
show(evaluate.favourite_acca_test(bt, S))
from scipy.stats import binomtest
b_acc, b_sl, _ = evaluate.acca_backtest(bt, {**S, "acca_markets": ["win"], "acca_leg_min_prob": S["banger_slips_min_prob"]})
if b_acc["slips"]:
    n_tried = len(pd.read_csv(COMBO_FILE))
    p = binomtest(b_acc["won"], b_acc["slips"], b_acc["break_even"], alternative="greater").pvalue
    top = b_sl.sort_values("profit", ascending=False).head(10)["profit"].sum() / b_sl["profit"].sum()
    print(f"Banger acca: won {b_acc['won']} of {b_acc['slips']} = {b_acc['hit']:.1%} | the model expected {b_acc['expected_hit']:.1%} | break-even {b_acc['break_even']:.1%}")
    print(f"Chance of this result if it really were only break-even: p = {p:.3f}. After allowing for {n_tried} combinations tried: p = {min(1, p * n_tried):.2f} (below 0.05 would be significant)")
    print(f"The 10 biggest wins make {top:.0%} of its profit" + (" (over 100% = without them it would lose)" if top > 1 else ""))
    b_sl["fold"] = b_sl["date"].apply(lambda d: f"{d.year if d.month >= 7 else d.year - 1}/{(d.year + 1 if d.month >= 7 else d.year) % 100:02d}")
    show(b_sl.groupby("fold").agg(slips=("won", "size"), hit=("won", "mean"), avg_odds=("odds", "mean"), roi=("profit", "mean")).reset_index())

### My picks: the exact rule, tested

Each day, only when the model has them: 1 long acca (banger, else strong) and up to 2 safer accas at about 2.3 odds, with no match used twice. Built exactly like the live tab, at the same stake on every slip.

**This is about break-even, not a sure profit.** It is here because it is the most disciplined way to use the model, not because it wins. I chose it after seeing the tests above, so the real result can be worse. The reasons not to trust acca numbers, above, apply here too.

In [ ]:
mp = evaluate.my_picks_test(pipeline.load_backtest(), S)
show(mp["table"])
print(f"Played on {mp['days']:,} of {mp['all_days']:,} match days. Worst losing runs: {mp['run_long']} long slips in a row, {mp['run_safer']} safer slips in a row.")

### The price you take decides the result

The same strategy at three prices: average of all bookmakers, one real bookmaker (Bet365), and the best price on the
whole market. Treat "best price" as the upper limit: it means always having an account at the bookmaker that pays the most.
Compare your own bookmaker's odds with the `Odds` and `Best odds` columns in the Excel file.

In [ ]:
rows = []
for name, price in (("Average odds", "avg"), ("Bet365", "bet365"), ("Best on the market", "best")):
    a, _, _ = evaluate.acca_backtest(bt, S, price)
    rows.append({"Price": name, "slips": a["slips"], "hit": a["hit"], "break_even": a["break_even"], "avg_odds": a["avg_odds"], "roi": a["roi"]})
show(pd.DataFrame(rows))

rows = []
MIX = ["win", "double_chance", "over_under"]
base = {**S, "acca_leagues": "main", "acca_leg_min_prob": 0.50, "acca_markets": MIX}
for name, s2 in (("Mixed, legs 50%+ (default)", base),
                 ("Mixed, legs 55%+", {**base, "acca_leg_min_prob": 0.55}),
                 ("Mixed, legs 60%+", {**base, "acca_leg_min_prob": 0.60}),
                 ("Mixed, no double chance", {**base, "acca_markets": ["win", "over_under"]}),
                 ("Mixed, all 38 leagues", {**base, "acca_leagues": "all"}),
                 ("Favourites only, legs 50%+", {**base, "acca_markets": ["win"]}),
                 ("Favourites only, legs 60%+", {**base, "acca_markets": ["win"], "acca_leg_min_prob": 0.60}),
                 ("Strong acca: favourites only, legs 65%+", {**base, "acca_markets": ["win"], "acca_leg_min_prob": 0.65}),
                 ("Banger acca: favourites only, legs 70%+", {**base, "acca_markets": ["win"], "acca_leg_min_prob": 0.70}),
                 ("Strong acca, all 38 leagues", {**base, "acca_markets": ["win"], "acca_leg_min_prob": 0.65, "acca_leagues": "all"})):
    a, sl, dd = evaluate.acca_backtest(bt, s2)
    b, sb, _ = evaluate.acca_backtest(bt, s2, "best")
    wk = pd.to_datetime(dd["date"]).dt.day_of_week >= 5
    lo, hi = evaluate.roi_range(sl["profit"], sl["date"])
    rows.append({"Rule": name, "Slips a day": round(a["slips"] / a["days"], 1), "Weekends with 5 slips": f"{(dd.loc[wk, 'slips'] == 5).mean():.0%}",
                 "hit": a["hit"], "break_even": a["break_even"], "roi": a["roi"], "95% range": f"{lo:+.1%} to {hi:+.1%}", "ROI at best odds": f"{b['roi']:+.1%}"})
show(pd.DataFrame(rows))

## 8. Your bet tracker
Every suggestion is logged in `output/bet_tracker.xlsx` and settled automatically. Mark `Played = Y` on the bets you really placed.

In [ ]:
show(tracker.summary(tracker.load()))

## 9. How it works and why each part was chosen

**Data.** football-data.co.uk is free and has results and bookmaker odds for 38 leagues, 22 seasons for the main leagues. Team names are
cleaned, so a club that changed its name or spelling keeps its history. A saved copy is in `data_snapshot/`.

**Features (what the model reads).** Only information known before kickoff:
Elo strength, form over 5 and 10 matches, home form at home and away form away, goals for and against, shots on target,
attack and defence strength, expected goals, head to head, rest days, league trends and bookmaker odds with the margin removed.

**Why these models:**

| Model | Why it is here |
|---|---|
| Logistic Regression | Simple and stable. The baseline: a fancier model must beat it. |
| XGBoost | Finds patterns that are not straight lines, like form only mattering between close teams. |
| LightGBM | Same family, built differently, so its mistakes are different and mixing the two reduces them. |
| Neural Network | A third way of learning. On this data it is the weakest alone, but the stack decides how much to trust it. |
| Stack (meta model) | Learns on the newest 15% of matches how much to trust each model. This also calibrates: 70% should mean 70%. |

**Tuning.** Optuna tried 30 settings each for XGBoost and LightGBM, on seasons before 2018/19 only, so the backtest never saw them.

**No leak test.** Every future result was scrambled at several dates: no feature before those dates changed.

**Why matches need odds.** The market is the strongest single signal in football. Matches without odds yet use a second model, marked `no odds`, which is weaker.

## 10. Tune the models again (optional, about 20 minutes)

In [ ]:
RUN_TUNING = False
if RUN_TUNING:
    from p90 import tuning
    from p90.features import build_features
    from p90.backtest import past_warmup
    res = pipeline.data.load_results(S["leagues"], S["first_season"])
    print(tuning.tune(past_warmup(build_features(res))))
    pipeline.get_models(build_features(res), S, force=True)

## 11. Honest notes

- **Football is random.** Even the bookmakers' favourite wins only about half of all matches. Nobody gets 60% on *every* match.
  The model earns its accuracy by knowing **which** matches are safer: the higher the confidence, the higher the hit rate (part 4).
- **Better probabilities, but no proven profit.** The model's probabilities are measurably better than the bookmaker's, but only by a tiny amount (part 4). The bookmaker's margin is much bigger, so flat betting on picks loses a little, and the value bets and slips are not proven winners (parts 6 and 7).
- **All 38 leagues together sit near 50%.** The bookmaker's favourite gets the same. Top leagues are higher, lower divisions are lower (part 4, by league).
- **Backtest is not a promise.** Past seasons show how the method behaves, not what will happen next week.
- **Odds differ by league in the data.** The 22 main leagues have odds from before the match. The 16 extra leagues only have closing odds in their history, which are harder to beat. Live, all leagues use odds from before the match.
- **Results arrive with a delay.** football-data.co.uk updates about twice a week, so a match played yesterday may not be counted yet in form and Elo, and bets may settle a few days later.
- **The main acca loses at average odds.** Mixing cost a little compared with favourites only (part 7), but gives all 5 slips on more days. Double chance prices in the test are worked out from the win odds.
- **No acca has proven profit.** The strong and banger accas made a small profit in the test, but the range includes a loss, and other library versions moved the banger acca from +7.2% to +27.6% with the same model and data. Do not plan on any acca number.
- **Other leagues do worse in accas.** Strong accas in all 38 leagues lost clearly (part 7), so accas use the 22 main leagues by default. The other leagues' old odds are closing odds, which is a harder test.
- **The other acca: accuracy is tested, profit is not.** Their old odds are not in the free data. More bets mean more slips, not proven profit. An estimated price is not a real price: check it at your bookmaker.
- **National teams: accuracy tested, profit not.** The free data has no old odds for national teams, so their accas can't be tested for profit, and they get no value bets. Friendlies are always WEAK. National team matches only come in the international windows, and only with the API-Football key.
- **1up and 2up are not included.** The free data has no goal minutes, so they can't be predicted or tested.
- **Bet only what you can afford to lose.**

### What differs from the plan, and why

| Plan | Delivered | Why |
|---|---|---|
| ClubElo ratings | Own Elo ratings, built from the results | ClubElo covers Europe only. Own Elo works for all 38 leagues and is computed the same way everywhere. |
| Odds movement as a feature | Not used | Closing odds are only known at kickoff. Training on them would be a data leak, and the live system could not have them in the morning. |
| Injuries in the model | In, as a small correction | Tested on 6,248 past matches: each missing player is worth about 0.3 points, because the odds already price it. It helped in one test season and hurt slightly in the other. It is not in the backtest numbers. Turn it off with `injury_adjust` in `settings.json`. |
| Lineups in the model | Shown, not modelled | Lineups come out under an hour before kickoff, one request per match, and the free plan has no way to get them for past seasons at this scale. Use part 3 to check them yourself. |